# RNN/LSTM

RNN é uma rede neural com memória, usada quando a ordem dos dados importa.

A LSTM é uma evolução da RNN. Ela foi criada para lidar melhor com dependências longas. Otimiza memória, tenta resolver a pergunta: “o que eu devo manter na memória e o que posso descartar?”.



In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:


# Base pequena de exemplo
entrada_textos = [
    "hello",
    "good morning",
    "good night",
    "how are you",
    "thank you",
    "i love you",
    "see you later"
]

saida_textos = [
    "<start> olá <end>",
    "<start> bom dia <end>",
    "<start> boa noite <end>",
    "<start> como você está <end>",
    "<start> obrigado <end>",
    "<start> eu te amo <end>",
    "<start> até mais tarde <end>"
]

In [4]:
# Tokenização
tokenizer_entrada = Tokenizer()
tokenizer_saida = Tokenizer(filters='')

tokenizer_entrada.fit_on_texts(entrada_textos)
tokenizer_saida.fit_on_texts(saida_textos)

entrada_seq = tokenizer_entrada.texts_to_sequences(entrada_textos)
saida_seq = tokenizer_saida.texts_to_sequences(saida_textos)

max_entrada = max(len(seq) for seq in entrada_seq)
max_saida = max(len(seq) for seq in saida_seq)

entrada_seq = pad_sequences(entrada_seq, maxlen=max_entrada, padding="post")
saida_seq = pad_sequences(saida_seq, maxlen=max_saida, padding="post")

vocab_entrada = len(tokenizer_entrada.word_index) + 1
vocab_saida = len(tokenizer_saida.word_index) + 1

# Entrada e saída do decoder
decoder_input = saida_seq[:, :-1]
decoder_output = saida_seq[:, 1:]

In [5]:
# Modelo Encoder-Decoder com LSTM
latent_dim = 64

# Configuração do Encoder
encoder_inputs = Input(shape=(max_entrada,))
encoder_emb = Embedding(vocab_entrada, latent_dim)(encoder_inputs)
encoder_lstm, state_h, state_c = LSTM(latent_dim, return_state=True)(encoder_emb)

# Configuração do Decoder
decoder_inputs = Input(shape=(max_saida - 1,))
decoder_emb = Embedding(vocab_saida, latent_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True)(decoder_emb, initial_state=[state_h, state_c])
decoder_dense = Dense(vocab_saida, activation="softmax")(decoder_lstm)

# Definição e Compilação do Modelo
model = Model([encoder_inputs, decoder_inputs], decoder_dense)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Treinamento
model.fit(
    [entrada_seq, decoder_input],
    np.expand_dims(decoder_output, -1),
    epochs=300,
    verbose=0
)

print("Modelo RNN/LSTM treinado.")

Modelo RNN/LSTM treinado.


In [6]:
def traduzir(frase):
    entrada = tokenizer_entrada.texts_to_sequences([frase])
    entrada = pad_sequences(entrada, maxlen=max_entrada, padding="post")

    decoder_seq = tokenizer_saida.texts_to_sequences(["<start>"])[0]
    decoder_seq = pad_sequences([decoder_seq], maxlen=max_saida - 1, padding="post")

    pred = model.predict([entrada, decoder_seq], verbose=0)
    pred_indices = np.argmax(pred[0], axis=1)

    palavras = []
    index_word = tokenizer_saida.index_word

    for idx in pred_indices:
        palavra = index_word.get(idx, "")
        if palavra == "<end>":
            break
        if palavra not in ["<start>", ""]:
            palavras.append(palavra)

    return " ".join(palavras)

print(traduzir("good morning"))
print(traduzir("thank you"))
print(traduzir("good night"))
print(traduzir("world"))

bom dia
obrigado
boa noite
olá


In [7]:
print(traduzir("i love you"))

eu te amo amo


# Transformer



In [8]:
!pip install transformers sentencepiece -q

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-ROMANCE"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def traduzir(texto):
    inputs = tokenizer(texto, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(traduzir("We need your help Starfox!"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/779k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/799k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Necesitamos su ayuda Starfox!


In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Modelo específico e robusto para tradução de Inglês para Português
model_name = "Helsinki-NLP/opus-mt-tc-big-en-pt"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def traduzir(texto):
    inputs = tokenizer(texto, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(traduzir("So close, no matter how far"))
print(traduzir("Couldn't be much more from the heart"))
print(traduzir("Forever trusting who we are"))
print(traduzir("And nothing else matters"))
print(traduzir("Never opened myself this way"))
print(traduzir("Life is ours, we live it our way"))
print(traduzir("All these words, I don't just say"))
print(traduzir("And nothing else matters"))
print(traduzir("Trust I seek and I find in you"))
print(traduzir("Every day for us something new"))
print(traduzir("Open mind for a different view"))
print(traduzir("And nothing else matters"))
print(traduzir("Never cared for what they do"))
print(traduzir("Never cared for what they know"))
print(traduzir("But I know"))
print(traduzir("So close, no matter how far"))
print(traduzir("It couldn't be much more from the heart"))
print(traduzir("Forever trusting who we are"))
print(traduzir("And nothing else matters"))
print(traduzir("Never cared for what they do"))
print(traduzir("Never cared for what they know"))
print(traduzir("But I know"))
print(traduzir("I never opened myself this way"))
print(traduzir("Life is ours, we live it our way"))
print(traduzir("All these words, I don't just say"))
print(traduzir("And nothing else matters"))
print(traduzir("Trust I seek and I find in you"))
print(traduzir("Every day for us something new"))
print(traduzir("Open mind for a different view"))
print(traduzir("And nothing else matters"))
print(traduzir("Never cared for what they say"))
print(traduzir("Never cared for games they play"))
print(traduzir("Never cared for what they do"))
print(traduzir("Never cared for what they know"))
print(traduzir("And I know, yeah, yeah"))
print(traduzir("So close, no matter how far"))
print(traduzir("Couldn't be much more from the heart"))
print(traduzir("Forever trusting who we are"))
print(traduzir("No, nothing else matters"))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/803k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/825k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/465M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Tão perto, não importa o quão longe
Não poderia ser muito mais do coração
Para sempre confiando em quem somos
E nada mais importa
Nunca me abri assim
A vida é nossa, vivemos do nosso jeito
Todas essas palavras, eu não digo apenas
E nada mais importa
Eu procuro e encontro confiança em você
Todos os dias para nós algo novo
Mente aberta para uma visão diferente
E nada mais importa
Nunca se importou com o que eles fazem
Nunca se importou com o que eles sabem
Mas eu sei
Tão perto, não importa o quão longe
Não poderia ser muito mais do coração
Para sempre confiando em quem somos
E nada mais importa
Nunca se importou com o que eles fazem
Nunca se importou com o que eles sabem
Mas eu sei
Eu nunca me abri dessa maneira
A vida é nossa, vivemos do nosso jeito
Todas essas palavras, eu não digo apenas
E nada mais importa
Eu procuro e encontro confiança em você
Todos os dias para nós algo novo
Mente aberta para uma visão diferente
E nada mais importa
Nunca se importaram com o que dizem
Nunca se impo

Análsie de Sentimentos com NLP

In [5]:
#INSTALAÇÃO DAS BIBLIOTECAS
!pip install deep_translator textblob
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.5 MB/s eta 0:00:00


In [6]:

#IMPORTAÇÃO DAS BIBLIOTECAS
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

#INICIALIZAÇÃO DO ANALISADOR DE SENTIMENTO
analisador = SentimentIntensityAnalyzer()

#DEFINIÇÃO DA FUNÇÃO DE ANÁLISE DE SENTIMENTO
def analisar_sentimento_vader(texto):
    texto_ingles = GoogleTranslator(source='auto', target='en').translate(texto)

    #CÁLCULO DO SENTIMENTO
    sentimento = analisador.polarity_scores(texto_ingles)

    #CLASSIFICAÇÃO DO SENTIMENTO
    if sentimento['compound'] >= 0.05:
        return "Positivo"
    elif sentimento['compound'] <= -0.05:
        return "Negativo"
    else:
        return "Neutro"

# TESTE DA FUNÇÃO
mensagem = "Estou muito feliz com o serviço do chatbot!"
print(analisar_sentimento_vader(mensagem))

Positivo


ROBERTA

In [11]:
from transformers import pipeline

# Criar pipeline de análise de sentimento multilíngue
analise_sentimento = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment")

# Teste
mensagem1 = "Jogaço, tenho 32 anos e gostei muito, minha namorada tbm se divertiu, o jogo atende tanto a iniciantes por ser bem instrutivo como a fãs"
resultado1 = analise_sentimento(mensagem1)
print(resultado1)

mensagem2 = "Tudo ok mas parece que o jogo já foi usado pois os pontos de ouro pra resgate já foi resgatado antes de mim... alguém já usou antes."
resultado2 = analise_sentimento(mensagem2)
print(resultado2)

mensagem3 = "Produto não veio em português como estava na descrição do produto."
resultado3 = analise_sentimento(mensagem3)
print(resultado3)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'positive', 'score': 0.9117110967636108}]
[{'label': 'neutral', 'score': 0.5551767349243164}]
[{'label': 'neutral', 'score': 0.5355968475341797}]


BERTimbau

In [12]:
from transformers import pipeline

# CARREGA O MODELO ESCOLHIDO
analise_sentimento = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

# TESTE COM TRÊS FRASES
mensagem1 = "Jogaço, tenho 32 anos e gostei muito, minha namorada tbm se divertiu, o jogo atende tanto a iniciantes por ser bem instrutivo como a fãs"
resultado1 = analise_sentimento(mensagem1)
print(resultado1)

mensagem2 = "Tudo ok mas parece que o jogo já foi usado pois os pontos de ouro pra resgate já foi resgatado antes de mim... alguém já usou antes."
resultado2 = analise_sentimento(mensagem2)
print(resultado2)

mensagem3 = "Produto não veio em português como estava na descrição do produto."
resultado3 = analise_sentimento(mensagem3)
print(resultado3)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'label': '5 stars', 'score': 0.6145632863044739}]
[{'label': '3 stars', 'score': 0.575551450252533}]
[{'label': '1 star', 'score': 0.581873893737793}]
